<a href="https://www.kaggle.com/code/sanchitgarg999/multimodel?scriptVersionId=309845635" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Cell 1 — Imports & Session Timer


In [1]:
import os, time, json, subprocess, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SESSION_START     = time.time()
SESSION_LIMIT_HRS = 11.0

def hours_elapsed():
    return (time.time() - SESSION_START) / 3600

def session_ok(buffer_hrs=0.5):
    """Returns False when within buffer_hrs of the 11-hr Kaggle limit."""
    return hours_elapsed() < (SESSION_LIMIT_HRS - buffer_hrs)

print(f'Session limit: {SESSION_LIMIT_HRS} hrs')


Device: cuda
GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
Session limit: 11.0 hrs


## Cell 2 — Streaming Multi-Part Extraction
No combined zip is ever written to disk. Peak disk usage ~6–8 GB per modality.


In [2]:
import io, zipfile, shutil
from pathlib import Path

# ── Locate the 9 zip parts (recursive — works with any dataset layout) ────
INPUT_BASE = Path('/kaggle/input')

parts = sorted(
    [
        f for f in INPUT_BASE.rglob('*')
        if f.is_file()
        and f.name.upper().startswith('MMRDR.ZIP.')
        and f.suffix.lstrip('.').isdigit()
    ],
    key=lambda p: int(p.suffix.lstrip('.'))
)

print(f'Found {len(parts)} zip parts:')
for p in parts:
    print(f'  {p}  ({p.stat().st_size / 1e6:.1f} MB)')

if len(parts) == 0:
    print('\nDebug — all files under /kaggle/input:')
    for f in sorted(INPUT_BASE.rglob('*')):
        if f.is_file():
            print(f'  {f}')

assert len(parts) == 9, (
    f'Expected 9 parts, found {len(parts)}. '
    f'Attach the dataset containing MMRDR.zip.001 to MMRDR.zip.009 via Add Data.'
)

total_zip_gb = sum(p.stat().st_size for p in parts) / 1e9
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'Total compressed : {total_zip_gb:.2f} GB')
print(f'Free disk space  : {free_gb:.2f} GB')
print('Strategy         : streaming extraction (no combined zip on disk)')


class MultiPartReader(io.RawIOBase):
    """
    Treats N split-zip part files as a single, seekable byte stream.
    No data is ever copied to disk.
    """
    def __init__(self, paths):
        self._paths   = list(paths)
        self._sizes   = [p.stat().st_size for p in self._paths]
        self._offsets = []
        off = 0
        for s in self._sizes:
            self._offsets.append(off)
            off += s
        self._total = off
        self._pos   = 0
        self._fh    = None
        self._cur   = -1

    def readable(self):  return True
    def seekable(self):  return True

    def readinto(self, b):
        if self._pos >= self._total:
            return 0
        n_want, n_read = len(b), 0
        while n_want > 0 and self._pos < self._total:
            idx      = self._part_for(self._pos)
            part_off = self._pos - self._offsets[idx]
            avail    = self._sizes[idx] - part_off
            to_read  = min(n_want, avail)
            self._open_part(idx)
            self._fh.seek(part_off)
            chunk  = self._fh.read(to_read)
            actual = len(chunk)
            if actual == 0:
                break
            b[n_read:n_read + actual] = chunk
            n_read    += actual
            n_want    -= actual
            self._pos += actual
        return n_read

    def seek(self, pos, whence=0):
        if   whence == 0: self._pos = pos
        elif whence == 1: self._pos = self._pos + pos
        elif whence == 2: self._pos = self._total + pos
        self._pos = max(0, min(self._pos, self._total))
        return self._pos

    def tell(self): return self._pos

    def _part_for(self, pos):
        lo, hi = 0, len(self._offsets) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self._offsets[mid] <= pos: lo = mid
            else:                         hi = mid - 1
        return lo

    def _open_part(self, idx):
        if self._cur != idx:
            if self._fh: self._fh.close()
            self._fh  = open(self._paths[idx], 'rb')
            self._cur = idx

    def close(self):
        if self._fh: self._fh.close()
        super().close()


EXTRACT_BASE = Path('/kaggle/working/mmrdr_extracted')
EXTRACT_BASE.mkdir(parents=True, exist_ok=True)
_READ_BUFFER = 4 << 20  # 4 MB


def _open_virtual_zip():
    raw = MultiPartReader(parts)
    buf = io.BufferedReader(raw, buffer_size=_READ_BUFFER)
    return zipfile.ZipFile(buf, 'r')


def _zip_prefix_for(modality_name):
    with _open_virtual_zip() as zf:
        members = zf.namelist()
    for prefix in [f'MMRDR/{modality_name}/', f'{modality_name}/']:
        if any(m.startswith(prefix) for m in members):
            return prefix
    top = sorted({m.split('/')[0] for m in members if m})
    raise FileNotFoundError(
        f'Cannot find {modality_name!r} in zip. Top-level folders: {top}'
    )


def extract_modality(modality_name, dest_dir=None):
    """Stream-extract one modality. The combined zip is NEVER written to disk."""
    dest_dir = Path(dest_dir or EXTRACT_BASE)
    dest_dir.mkdir(parents=True, exist_ok=True)
    prefix = _zip_prefix_for(modality_name)
    t0 = time.time()
    with _open_virtual_zip() as zf:
        members = [m for m in zf.namelist() if m.startswith(prefix)]
        n = len(members)
        print(f'  Extracting {n:,} entries for {modality_name}...')
        for i, member in enumerate(members, 1):
            zf.extract(member, dest_dir)
            if i % 500 == 0 or i == n:
                free = shutil.disk_usage('/kaggle/working').free / 1e9
                print(f'    {i:,}/{n:,} done  |  disk free: {free:.1f} GB', end='\r')
    print()
    modality_dir = dest_dir / prefix.rstrip('/')
    print(f'  Done in {time.time()-t0:.0f}s  ->  {modality_dir}')
    return modality_dir


def free_modality_images(modality_dir):
    """Delete images to reclaim disk space after training. Keeps CSVs."""
    img_dir = Path(modality_dir) / 'img'
    if img_dir.exists():
        shutil.rmtree(img_dir)
        free = shutil.disk_usage('/kaggle/working').free / 1e9
        print(f'  Freed {modality_dir.name}/img  |  disk free: {free:.1f} GB')
    else:
        print(f'  {img_dir} already removed.')


# Quick sanity: scan zip table-of-contents without extracting anything
print('\nScanning zip contents (no extraction yet)...')
with _open_virtual_zip() as zf:
    all_members = zf.namelist()
top_level = sorted({m.split('/')[0] for m in all_members if m})
print(f'  Total zip entries : {len(all_members):,}')
print(f'  Top-level folders : {top_level}')
print('\nReady. Modalities will be extracted one at a time in the task cells.')


Found 9 zip parts:
  /kaggle/input/datasets/sanchitgarg999/mmrdr001/MMRDR.zip.001  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr002/MMRDR.zip.002  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr003/MMRDR.zip.003  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr004/MMRDR.zip.004  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr005/MMRDR.zip.005  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr006/MMRDR.zip.006  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr007/MMRDR.zip.007  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr008/MMRDR.zip.008  (2097.2 MB)
  /kaggle/input/datasets/sanchitgarg999/mmrdr009/MMRDR.zip.009  (1833.1 MB)
Total compressed : 18.61 GB
Free disk space  : 20.94 GB
Strategy         : streaming extraction (no combined zip on disk)

Scanning zip contents (no extraction yet)...
  Total zip entries : 24,469
  Top-level folders : ['MMRDR-CFP', 'MMRDR-OCT', 'MMRDR-UWF']

Ready. Modalities will be extracted o

## Cell 3 — Configuration


In [3]:
OUTPUT_DIR = Path('/kaggle/working/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'img_size'    : 512,
    'batch_size'  : 32,
    'num_workers' : 4,
    'lr'          : 1e-4,
    'weight_decay': 1e-4,
    'epochs'      : 50,
    'val_split'   : 0.125,  # 7:1 train/val per paper
    'patience'    : 10,
}

LESION_NAMES   = ['Microaneurysm','Hard Exudate','Intraretinal Hemorrhage',
                   'VB/IRMA','Neovascularization','Vitreous Hemorrhage','Retinal Detachment']
DR_GRADE_NAMES = ['No DR','Mild NPDR','Moderate NPDR','Severe NPDR','PDR']
DME_NAMES      = ['No DME','NCI DME','CI DME']

print('Configuration loaded.')
print(f'Image size : {CFG["img_size"]}x{CFG["img_size"]}')
print(f'Batch size : {CFG["batch_size"]}')
print(f'Max epochs : {CFG["epochs"]}')
print(f'Output dir : {OUTPUT_DIR}')


Configuration loaded.
Image size : 512x512
Batch size : 32
Max epochs : 50
Output dir : /kaggle/working/results


## Cell 4 — Data Transforms


In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined.')


Transforms defined.


## Cell 5 — Dataset Classes


In [5]:
class MMRDRGradingDataset(Dataset):
    """Single-label: CFP/UWF DR grading (5-class) and OCT DME (3-class)."""
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        # FIX: Handle image paths that may already contain 'img/' prefix,
        # or be bare filenames. Resolve to correct absolute path either way.
        img_val  = str(row['image'])
        img_path = self.img_dir / img_val  # try as-is first
        if not img_path.exists():
            # Try with 'img/' subdir prepended (bare filename case)
            img_path = self.img_dir / 'img' / Path(img_val).name
        image    = Image.open(img_path).convert('RGB')
        label    = int(row['grade'])
        if self.transform:
            image = self.transform(image)
        return image, label


class MMRDRLesionDataset(Dataset):
    """Multi-label: CFP/UWF lesion classification (7 binary labels)."""
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        # FIX: same dual-path resolution as GradingDataset above
        img_val  = str(row['image'])
        img_path = self.img_dir / img_val
        if not img_path.exists():
            img_path = self.img_dir / 'img' / Path(img_val).name
        image    = Image.open(img_path).convert('RGB')
        # FIX: parse lesion string safely instead of eval(); return float32 directly
        lesion_str = str(row['lesion']).strip().strip('[]')
        label = torch.tensor([float(x) for x in lesion_str.split(',')], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label


print('Dataset classes defined.')


Dataset classes defined.


## Cell 6 — CSV Loader & Train/Val Split


In [6]:
def load_csv_splits(modality_dir, csv_filename):
    """
    Reads annotation CSV, splits train rows into train/val (7:1).
    Images prefixed 'tr' = training, 'ts' = testing.

    FIX: The 'image' column may store bare filenames ('tr000001.jpg'),
    paths with a subfolder ('img/tr000001.jpg'), or other variants.
    We extract only the basename before checking the prefix so the
    filter works regardless of how the CSV was written.
    """
    df = pd.read_csv(Path(modality_dir) / csv_filename)

    # Normalise: keep only the filename stem for prefix detection
    basenames = df['image'].apply(lambda p: Path(str(p)).name)

    train_mask = basenames.str.startswith('tr')
    test_mask  = basenames.str.startswith('ts')

    # Fallback: if neither prefix matches, warn and do an 80/20 split
    if train_mask.sum() == 0 and test_mask.sum() == 0:
        print(f'  WARNING: no tr/ts prefix found in image column. '
              f'Sample values: {df["image"].head(3).tolist()}')
        print(f'  Falling back to 80/20 random split on all {len(df):,} rows.')
        train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
    elif train_mask.sum() == 0:
        # All rows are test — use them for both (edge-case guard)
        print('  WARNING: no training rows found; using test set for all splits.')
        train_df = df[test_mask].copy()
        test_df  = df[test_mask].copy()
    else:
        train_df = df[train_mask].copy()
        test_df  = df[test_mask].copy()

    if len(train_df) == 0:
        raise RuntimeError(
            f"Training set is empty after filtering. CSV path: "
            f"{Path(modality_dir) / csv_filename}\n"
            f"First 5 image values: {df['image'].head().tolist()}"
        )

    # Stratify on grade only when the column exists and has enough samples per class
    strat = None
    if 'grade' in train_df.columns:
        min_class_count = train_df['grade'].value_counts().min()
        n_val = max(1, int(len(train_df) * CFG['val_split']))
        n_classes = train_df['grade'].nunique()
        if min_class_count >= 2 and n_val >= n_classes:
            strat = train_df['grade']
        else:
            print(f'  NOTE: skipping stratify (min class count={min_class_count}, '
                  f'n_val={n_val}, n_classes={n_classes})')

    train_df, val_df = train_test_split(
        train_df, test_size=CFG['val_split'], random_state=SEED, stratify=strat
    )
    print(f'  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
    return train_df, val_df, test_df


print('CSV loader defined.')


CSV loader defined.


## Cell 7 — ResNet-50 Model (Offline Weights) & Loss Functions


In [7]:
# FIX: Only one build_resnet50 definition — loads weights from Kaggle dataset.
# The previous notebook had two definitions: the first tried to download from the
# internet (fails with internet off), and the second correctly loaded from disk
# but silently overwrote the first. Now there is only the correct offline version.

# Path to the uploaded ResNet-50 weights dataset
# Adjust this path if your Kaggle dataset slug differs
_RESNET50_WEIGHTS = next(
    Path('/kaggle/input').rglob('resnet50*.pth'),
    None
)
if _RESNET50_WEIGHTS is None:
    # Fallback: try .pt extension
    _RESNET50_WEIGHTS = next(Path('/kaggle/input').rglob('resnet50*.pt'), None)

assert _RESNET50_WEIGHTS is not None, (
    'ResNet-50 weights file not found under /kaggle/input. '
    'Add your resnet50 weights dataset via Add Data and make sure the '
    'file is named resnet50*.pth or resnet50*.pt'
)
print(f'ResNet-50 weights: {_RESNET50_WEIGHTS}')


def build_resnet50(num_classes):
    """Load offline ResNet-50 ImageNet weights, replace FC for num_classes."""
    model = models.resnet50(weights=None)  # no internet download
    # FIX: weights_only=True suppresses FutureWarning in PyTorch >= 2.0
    state_dict = torch.load(_RESNET50_WEIGHTS, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


def get_loss(task):
    if task == 'lesion':
        return nn.BCEWithLogitsLoss()
    elif task == 'dme':
        # Weighted CE: NCI DME class has far fewer samples
        total = 1017 + 280 + 1641
        w = torch.tensor([total/1017, total/280, total/1641], dtype=torch.float32).to(DEVICE)
        return nn.CrossEntropyLoss(weight=w)
    else:
        return nn.CrossEntropyLoss()


print('Model builder defined.')


ResNet-50 weights: /kaggle/input/datasets/sanchitgarg999/resnet50-imagenet-weights/resnet50-11ad3fa6.pth
Model builder defined.


## Cell 8 — Training & Evaluation Functions


In [8]:
def train_one_epoch(model, loader, optimizer, criterion, task):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, task):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        if task == 'lesion':
            preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy().astype(int)
            all_preds.append(preds)
            all_labels.append(labels.numpy().astype(int))
        else:
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    if task == 'lesion':
        acc = accuracy_score(all_labels.flatten(), all_preds.flatten())
        f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    else:
        acc = accuracy_score(all_labels, all_preds)
        f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, f1


def train_task(model, train_loader, val_loader, task, task_name):
    """Full training loop with early stopping and cosine LR schedule."""
    criterion = get_loss(task)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
    best_val_acc = 0.0
    patience_ctr = 0
    best_ckpt    = OUTPUT_DIR / f'best_{task_name}.pth'

    for epoch in range(1, CFG['epochs'] + 1):
        if not session_ok():
            print(f'  Session limit approaching at epoch {epoch}. Stopping early.')
            break
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, task)
        val_acc, val_f1 = evaluate(model, val_loader, task)
        scheduler.step()
        print(f'  Ep {epoch:3d}/{CFG["epochs"]} | loss={train_loss:.4f} | '
              f'val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | '
              f'{time.time()-t0:.0f}s | session={hours_elapsed():.2f}h')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_ctr = 0
            torch.save(model.state_dict(), best_ckpt)
        else:
            patience_ctr += 1
            if patience_ctr >= CFG['patience']:
                print(f'  Early stopping at epoch {epoch}.')
                break
    print(f'  Best val acc: {best_val_acc:.4f}')
    return best_ckpt


print('Training functions defined.')


Training functions defined.


## Cell 9 — Results Logger & DataLoader Helper


In [9]:
RESULTS = {}

def log_results(name, acc, f1, paper_acc, paper_f1):
    RESULTS[name] = dict(acc=round(acc,4), f1=round(f1,4),
                         paper_acc=paper_acc, paper_f1=paper_f1,
                         acc_diff=round(acc-paper_acc,4), f1_diff=round(f1-paper_f1,4))
    print(f'  Accuracy: {acc:.4f}  (paper {paper_acc})  diff={acc-paper_acc:+.4f}')
    print(f'  F1 Score: {f1:.4f}  (paper {paper_f1})  diff={f1-paper_f1:+.4f}')


def make_loaders(train_df, val_df, test_df, img_dir, task):
    DS = MMRDRLesionDataset if task == 'lesion' else MMRDRGradingDataset
    tr = DataLoader(DS(train_df, img_dir, train_transform),
                    batch_size=CFG['batch_size'], shuffle=True,
                    num_workers=CFG['num_workers'], pin_memory=True)
    vl = DataLoader(DS(val_df,   img_dir, val_test_transform),
                    batch_size=CFG['batch_size'], shuffle=False,
                    num_workers=CFG['num_workers'], pin_memory=True)
    ts = DataLoader(DS(test_df,  img_dir, val_test_transform),
                    batch_size=CFG['batch_size'], shuffle=False,
                    num_workers=CFG['num_workers'], pin_memory=True)
    return tr, vl, ts


print('Logger and loader helper defined.')


Logger and loader helper defined.


## Tasks 1 & 2 — CFP: DR Grading + Lesion Classification


In [10]:
# ── Extract CFP (streaming — no combined zip written to disk) ───────────
print('='*60)
print('Extracting MMRDR-CFP...')
print('='*60)
CFP_DIR = extract_modality('MMRDR-CFP')
print(f'Disk free: {shutil.disk_usage("/kaggle/working").free/1e9:.1f} GB')

# ── Task 1: CFP DR Grading (5-class) ─────────────────────────────────────
print()
print('='*60)
print('TASK 1: CFP DR Grading (5-class)')
print(f'Session: {hours_elapsed():.2f}h')
print('='*60)

cfp_train, cfp_val, cfp_test = load_csv_splits(CFP_DIR, 'FP.csv')
tr_l, vl_l, ts_l = make_loaders(cfp_train, cfp_val, cfp_test, CFP_DIR, 'grading')
model   = build_resnet50(num_classes=5)
ckpt    = train_task(model, tr_l, vl_l, 'grading', 'cfp_grading')
model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
acc, f1 = evaluate(model, ts_l, 'grading')
log_results('CFP_Grading', acc, f1, paper_acc=0.823, paper_f1=0.702)
del model; torch.cuda.empty_cache()

# ── Task 2: CFP Lesion Classification (7-label) ───────────────────────────
print()
print('='*60)
print('TASK 2: CFP Lesion Classification (7-label)')
print(f'Session: {hours_elapsed():.2f}h')
print('='*60)

if not session_ok(buffer_hrs=1.5):
    print('Session limit approaching. Skipping.')
else:
    tr_l, vl_l, ts_l = make_loaders(cfp_train, cfp_val, cfp_test, CFP_DIR, 'lesion')
    model   = build_resnet50(num_classes=7)
    ckpt    = train_task(model, tr_l, vl_l, 'lesion', 'cfp_lesion')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    acc, f1 = evaluate(model, ts_l, 'lesion')
    log_results('CFP_Lesion', acc, f1, paper_acc=0.944, paper_f1=0.671)
    del model; torch.cuda.empty_cache()

# ── Free CFP images before extracting UWF ────────────────────────────────
print()
free_modality_images(CFP_DIR)


Extracting MMRDR-CFP...
  Extracting 11,121 entries for MMRDR-CFP...
    11,121/11,121 done  |  disk free: 13.3 GB
  Done in 65s  ->  /kaggle/working/mmrdr_extracted/MMRDR-CFP
Disk free: 13.3 GB

TASK 1: CFP DR Grading (5-class)
Session: 0.02h
  Train: 7,781 | Val: 1,112 | Test: 2,225
  Ep   1/50 | loss=0.7766 | val_acc=0.7761 | val_f1=0.6128 | 61s | session=0.04h
  Ep   2/50 | loss=0.5384 | val_acc=0.8147 | val_f1=0.6995 | 60s | session=0.05h
  Ep   3/50 | loss=0.4773 | val_acc=0.8318 | val_f1=0.7097 | 60s | session=0.07h
  Ep   4/50 | loss=0.4447 | val_acc=0.8363 | val_f1=0.7070 | 61s | session=0.09h
  Ep   5/50 | loss=0.4106 | val_acc=0.8453 | val_f1=0.7297 | 59s | session=0.10h
  Ep   6/50 | loss=0.3924 | val_acc=0.8156 | val_f1=0.6808 | 62s | session=0.12h
  Ep   7/50 | loss=0.3632 | val_acc=0.8264 | val_f1=0.7012 | 60s | session=0.14h
  Ep   8/50 | loss=0.3309 | val_acc=0.8435 | val_f1=0.7288 | 61s | session=0.15h
  Ep   9/50 | loss=0.3054 | val_acc=0.8345 | val_f1=0.7191 | 61s |

## Tasks 3 & 4 — UWF: DR Grading + Lesion Classification


In [11]:
# ── Extract UWF ──────────────────────────────────────────────────────────
print('='*60)
print('Extracting MMRDR-UWF...')
print('='*60)

if not session_ok(buffer_hrs=3.0):
    print('Session limit approaching. Skipping UWF tasks.')
else:
    UWF_DIR = extract_modality('MMRDR-UWF')
    print(f'Disk free: {shutil.disk_usage("/kaggle/working").free/1e9:.1f} GB')

    print()
    print('='*60)
    print('TASK 3: UWF DR Grading (5-class)')
    print(f'Session: {hours_elapsed():.2f}h')
    print('='*60)

    uwf_train, uwf_val, uwf_test = load_csv_splits(UWF_DIR, 'UWF.csv')
    tr_l, vl_l, ts_l = make_loaders(uwf_train, uwf_val, uwf_test, UWF_DIR, 'grading')
    model   = build_resnet50(num_classes=5)
    ckpt    = train_task(model, tr_l, vl_l, 'grading', 'uwf_grading')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    acc, f1 = evaluate(model, ts_l, 'grading')
    log_results('UWF_Grading', acc, f1, paper_acc=0.745, paper_f1=0.733)
    del model; torch.cuda.empty_cache()

    print()
    print('='*60)
    print('TASK 4: UWF Lesion Classification (7-label)')
    print(f'Session: {hours_elapsed():.2f}h')
    print('='*60)

    if not session_ok(buffer_hrs=1.5):
        print('Session limit approaching. Skipping.')
    else:
        tr_l, vl_l, ts_l = make_loaders(uwf_train, uwf_val, uwf_test, UWF_DIR, 'lesion')
        model   = build_resnet50(num_classes=7)
        ckpt    = train_task(model, tr_l, vl_l, 'lesion', 'uwf_lesion')
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        acc, f1 = evaluate(model, ts_l, 'lesion')
        log_results('UWF_Lesion', acc, f1, paper_acc=0.923, paper_f1=0.774)
        del model; torch.cuda.empty_cache()

    print()
    free_modality_images(UWF_DIR)


Extracting MMRDR-UWF...
  Extracting 10,407 entries for MMRDR-UWF...
    10,407/10,407 done  |  disk free: 10.5 GB
  Done in 96s  ->  /kaggle/working/mmrdr_extracted/MMRDR-UWF
Disk free: 10.5 GB

TASK 3: UWF DR Grading (5-class)
Session: 0.72h
  Train: 6,831 | Val: 976 | Test: 2,597
  Ep   1/50 | loss=0.9577 | val_acc=0.7029 | val_f1=0.6657 | 109s | session=0.75h
  Ep   2/50 | loss=0.7145 | val_acc=0.7223 | val_f1=0.6968 | 109s | session=0.78h
  Ep   3/50 | loss=0.6480 | val_acc=0.7039 | val_f1=0.6617 | 110s | session=0.81h
  Ep   4/50 | loss=0.6218 | val_acc=0.7213 | val_f1=0.6835 | 110s | session=0.85h
  Ep   5/50 | loss=0.5942 | val_acc=0.7520 | val_f1=0.7228 | 109s | session=0.88h
  Ep   6/50 | loss=0.5645 | val_acc=0.7582 | val_f1=0.7305 | 107s | session=0.91h
  Ep   7/50 | loss=0.5407 | val_acc=0.7213 | val_f1=0.6884 | 110s | session=0.94h
  Ep   8/50 | loss=0.5137 | val_acc=0.7408 | val_f1=0.7141 | 108s | session=0.97h
  Ep   9/50 | loss=0.4856 | val_acc=0.7684 | val_f1=0.7408 |

## Task 5 — OCT: DME Classification


In [12]:
# ── Extract OCT ──────────────────────────────────────────────────────────
print('='*60)
print('Extracting MMRDR-OCT...')
print('='*60)

if not session_ok(buffer_hrs=0.75):
    print('Session limit approaching. Skipping OCT.')
else:
    OCT_DIR = extract_modality('MMRDR-OCT')
    print(f'Disk free: {shutil.disk_usage("/kaggle/working").free/1e9:.1f} GB')

    print()
    print('='*60)
    print('TASK 5: OCT DME Classification (3-class)')
    print(f'Session: {hours_elapsed():.2f}h')
    print('='*60)

    oct_train, oct_val, oct_test = load_csv_splits(OCT_DIR, 'OCT.csv')
    tr_l, vl_l, ts_l = make_loaders(oct_train, oct_val, oct_test, OCT_DIR, 'dme')
    model   = build_resnet50(num_classes=3)
    ckpt    = train_task(model, tr_l, vl_l, 'dme', 'oct_dme')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    acc, f1 = evaluate(model, ts_l, 'dme')  # single-label 3-class
    log_results('OCT_DME', acc, f1, paper_acc=0.890, paper_f1=0.701)
    del model; torch.cuda.empty_cache()


Extracting MMRDR-OCT...
  Extracting 2,941 entries for MMRDR-OCT...
    2,941/2,941 done  |  disk free: 18.5 GB
  Done in 24s  ->  /kaggle/working/mmrdr_extracted/MMRDR-OCT
Disk free: 18.5 GB

TASK 5: OCT DME Classification (3-class)
Session: 2.09h
  Train: 2,079 | Val: 297 | Test: 562
  Ep   1/50 | loss=0.9248 | val_acc=0.7643 | val_f1=0.6085 | 11s | session=2.10h
  Ep   2/50 | loss=0.7188 | val_acc=0.7980 | val_f1=0.6605 | 11s | session=2.10h
  Ep   3/50 | loss=0.6219 | val_acc=0.8047 | val_f1=0.6830 | 11s | session=2.10h
  Ep   4/50 | loss=0.5585 | val_acc=0.7879 | val_f1=0.6890 | 11s | session=2.10h
  Ep   5/50 | loss=0.5131 | val_acc=0.8350 | val_f1=0.7136 | 11s | session=2.11h
  Ep   6/50 | loss=0.4720 | val_acc=0.8418 | val_f1=0.7288 | 11s | session=2.11h
  Ep   7/50 | loss=0.4586 | val_acc=0.8451 | val_f1=0.7509 | 11s | session=2.11h
  Ep   8/50 | loss=0.4355 | val_acc=0.8418 | val_f1=0.7286 | 11s | session=2.12h
  Ep   9/50 | loss=0.3791 | val_acc=0.8283 | val_f1=0.7029 | 11s 

## Final Results Summary


In [13]:
print('\n' + '='*70)
print('FINAL RESULTS — ResNet-50 vs Paper (Table 3)')
print('='*70)

rows = []
for name, m in RESULTS.items():
    rows.append({'Task': name,
                 'Our Acc': m['acc'],   'Paper Acc': m['paper_acc'], 'Acc Diff': f"{m['acc_diff']:+.4f}",
                 'Our F1' : m['f1'],    'Paper F1' : m['paper_f1'],  'F1 Diff' : f"{m['f1_diff']:+.4f}"})

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))

df_res.to_csv(OUTPUT_DIR / 'final_results.csv', index=False)
with open(OUTPUT_DIR / 'final_results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2)

print(f'\nTotal session time: {hours_elapsed():.2f} / {SESSION_LIMIT_HRS} hrs')
print(f'Results saved to  : {OUTPUT_DIR}')



FINAL RESULTS — ResNet-50 vs Paper (Table 3)
       Task  Our Acc  Paper Acc Acc Diff  Our F1  Paper F1 F1 Diff
CFP_Grading   0.8431      0.823  +0.0201  0.7382     0.702 +0.0362
 CFP_Lesion   0.9558      0.944  +0.0118  0.7127     0.671 +0.0417
UWF_Grading   0.7501      0.745  +0.0051  0.7335     0.733 +0.0005
 UWF_Lesion   0.9242      0.923  +0.0012  0.7566     0.774 -0.0174
    OCT_DME   0.9004      0.890  +0.0104  0.7606     0.701 +0.0596

Total session time: 2.17 / 11.0 hrs
Results saved to  : /kaggle/working/results
